In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Any, Tuple

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SEED = 42
set_seed(SEED)

# -------------------------
# 1) Load CSV and build splits
# -------------------------
def load_and_prepare(train_path="train.csv", test_path="test.csv", val_ratio=0.1):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # normalize column names if needed
    # expected: "Class Index", "Title", "Description"
    for df in [train_df, test_df]:
        # text
        df["Title"] = df["Title"].fillna("").astype(str)
        df["Description"] = df["Description"].fillna("").astype(str)
        df["text"] = (df["Title"] + " " + df["Description"]).str.strip()

        # label: 1..4 -> 0..3
        df["label"] = df["Class Index"].astype(int) - 1

    # split train into train/val
    train_dataset = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
    split = train_dataset.train_test_split(test_size=val_ratio, seed=SEED, stratify_by_column="label")
    ds_train = split["train"]
    ds_val = split["test"]

    ds_test = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)
    return ds_train, ds_val, ds_test


# -------------------------
# 2) Tokenization
# -------------------------
def tokenize_dataset(ds, tokenizer, max_length=256):
    def _tok(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_length,
        )
    return ds.map(_tok, batched=True, remove_columns=["text"])


# -------------------------
# 3) Metrics: accuracy + macro-f1
# -------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "macro_f1": f1}


# -------------------------
# 4) Train + Evaluate + Confusion Matrix
# -------------------------
@dataclass
class RunResult:
    model_name: str
    val_metrics: Dict[str, Any]
    test_metrics: Dict[str, Any]
    test_confusion: np.ndarray
    test_report: str

def run_experiment(
    model_ckpt: str,
    ds_train,
    ds_val,
    ds_test,
    num_labels=4,
    out_dir="runs",
    epochs=2,
    lr=2e-5,
    batch_size=16,
) -> RunResult:
    tokenizer = AutoTokenizer.from_pretrained(model_ckpt, use_fast=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_tok = tokenize_dataset(ds_train, tokenizer)
    val_tok = tokenize_dataset(ds_val, tokenizer)
    test_tok = tokenize_dataset(ds_test, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels)

    args = TrainingArguments(
        output_dir=f"{out_dir}/{model_ckpt.replace('/', '_')}",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        seed=SEED,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # validation metrics
    val_metrics = trainer.evaluate(eval_dataset=val_tok)

    # test metrics + confusion matrix
    test_pred = trainer.predict(test_tok)
    test_logits = test_pred.predictions
    test_labels = test_pred.label_ids
    test_preds = np.argmax(test_logits, axis=-1)

    test_metrics = {
        "accuracy": accuracy_score(test_labels, test_preds),
        "macro_f1": f1_score(test_labels, test_preds, average="macro"),
    }
    cm = confusion_matrix(test_labels, test_preds)

    # nice report
    report = classification_report(test_labels, test_preds, digits=4)

    return RunResult(
        model_name=model_ckpt,
        val_metrics=val_metrics,
        test_metrics=test_metrics,
        test_confusion=cm,
        test_report=report,
    )


def main():
    ds_train, ds_val, ds_test = load_and_prepare("train.csv", "test.csv", val_ratio=0.1)

    experiments = [
        # WordPiece
        "distilbert-base-uncased",
        # BPE (lightweight distilled RoBERTa)
        "distilroberta-base",
    ]

    results = []
    for ckpt in experiments:
        print(f"\n====================\nRunning: {ckpt}\n====================")
        res = run_experiment(
            model_ckpt=ckpt,
            ds_train=ds_train,
            ds_val=ds_val,
            ds_test=ds_test,
            epochs=2,          # 2~3 usually enough for AG News
            lr=2e-5,
            batch_size=16,
        )
        results.append(res)

        print("\n[Validation metrics]")
        print({k: float(v) for k, v in res.val_metrics.items() if k.startswith(("eval_", "accuracy", "macro_f1"))})

        print("\n[Test metrics]")
        print(res.test_metrics)

        print("\n[Test confusion matrix] (rows=true, cols=pred)")
        print(res.test_confusion)

        print("\n[Test classification report]")
        print(res.test_report)

    # summary
    print("\n====================\nSummary\n====================")
    for r in results:
        print(f"{r.model_name}: test_acc={r.test_metrics['accuracy']:.4f}, test_macro_f1={r.test_metrics['macro_f1']:.4f}")


if __name__ == "__main__":
    main()
